# Lab 11 · Go keyless with Microsoft Entra ID and RBAC

Call your model through AI Gateway with a Microsoft Entra ID access token instead of an API key. You grant yourself
the **AI Gateway Models User** role on the gateway, inspect the token, call the model with the OpenAI SDK, and prove
that access follows the role assignment: `401` for the wrong audience, `403` without the role, `200` with it. This
notebook is the Python track of the [lab guide](README.md): it grants the role by deploying
[infra/role-assignment.bicep](infra/role-assignment.bicep), the same template as the Bicep track.

**Deploy role assignment → Get token → Call keyless → Wrong audience → Remove role → Restore**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| AI Gateway Models User (`a6b68e3d-a1f0-4c60-8678-237c050cf4ea`) | Data-plane role that lets you list and invoke models. Assignable at the gateway or a model alias. |
| Access token for `https://azure-api.net/aigateway` | Proves who you are. The gateway validates it at its edge, then checks your role assignments. |
| `get_bearer_token_provider` | Gives the OpenAI SDK a fresh token before every request, so no key is involved. |
| Alias `gpt-4.1-mini` | The model from notebook 02 that you call. |
| [infra/role-assignment.bicep](infra/role-assignment.bicep) | Grants an AI Gateway role to an identity at the gateway, one alias, or one MCP server. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

| Role | Role ID | Grants |
| --- | --- | --- |
| AI Gateway User | `caad7a21-134e-43f4-a550-c5ffc17e051c` | List and invoke models, list and call MCP tools |
| AI Gateway Models User | `a6b68e3d-a1f0-4c60-8678-237c050cf4ea` | List and invoke models |
| AI Gateway Tools User | `dce03f8d-857f-4e40-a90d-67770fa6ecd2` | List and call tools within MCP servers |

![Identity boundaries: callers authenticate to the gateway with Entra ID, and the gateway authenticates to backends with its own identity](../assets/diagrams/identity-boundaries.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-4.1-mini` alias.
- Permission to assign roles on the gateway, such as **Owner** or **Role Based Access Control Administrator**.

Role changes take a few minutes to apply, so this notebook waits and retries. Expect it to take 10 to 20 minutes.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q --upgrade azure-identity requests openai

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. The cell also removes
`AI_GATEWAY_API_KEY` from the environment to prove that nothing in this notebook uses a key.

In [ ]:
import os
import sys
sys.path.insert(0, "../shared")   # helpers that every lab notebook shares
import utils

os.environ.pop("AI_GATEWAY_API_KEY", None)   # nothing in this notebook uses a key
lab   = utils.Lab("11-entra-rbac")
alias = "gpt-4.1-mini"   # model alias from notebook 02

### 1. Find your identity and check the model

Role assignments target an object ID. `lab.signed_in_identity` reads yours from your Azure CLI token (the lab guide
uses `az ad signed-in-user show`). The cell also confirms that notebook 02 published the alias.

In [ ]:
my_object_id, principal_type, my_name = lab.signed_in_identity()
print(f"👤 {my_name}: {my_object_id} ({principal_type})")

lab.require(f"{lab.workspace_id}/aliases/{alias}", "Run 02-first-model-call.ipynb first.")
print(f"✅ Alias {alias} exists")

### 2. Grant yourself a data-plane role

[infra/role-assignment.bicep](infra/role-assignment.bicep) assigns an AI Gateway role at the gateway, one alias, or
one MCP server. Deploy it with `role="models"` to grant yourself **AI Gateway Models User** on the gateway. These
roles grant **runtime** access only, not gateway administration. The assignment name is derived from the scope,
principal, and role, so a redeploy reuses the same assignment.

If you already hold the role on the gateway, the cell skips the deployment. It then lists the AI Gateway roles that
you hold on the gateway, including inherited ones.

In [ ]:
def grant():
    return lab.deploy("infra/role-assignment.bicep", gatewayName=lab.gateway_name, principalId=my_object_id,
                      principalType=principal_type, role="models")

def models_user_on_gateway():
    return [a["id"] for a in lab.role_assignments(my_object_id)
            if a["role"] == "AI Gateway Models User" and a["scope"].lower() == lab.gateway_id.lower()]

had_role_before = bool(models_user_on_gateway())
assignment = None if had_role_before else grant()
print("♻️ You already have" if had_role_before else "✅ Assigned", "AI Gateway Models User on the gateway")

mine = {a["id"].lower(): a for a in lab.role_assignments(my_object_id)}
if assignment:   # the list can lag a new assignment by minutes
    mine.setdefault(assignment["roleAssignmentId"].lower(),
                    {"role": "AI Gateway Models User", "scope": assignment["scopeId"]})
for a in mine.values():
    print(f"🔐 {a['role']:<24} {a['scope']}")

### 3. Get a token and look inside it

Request a token for the AI Gateway scope. Apps that run as a service principal, managed identity, or agent identity
use `/.default`, and it works for an Azure CLI sign-in too. The cell decodes the payload to show what the gateway
checks: `aud` is `https://azure-api.net/aigateway`, and `tid` plus `oid` identify you to per-caller policies such as
the token limits in notebook 06.

In [ ]:
token  = lab.token(utils.AIGW_SCOPE)
fields = {k: utils.claims(token).get(k) for k in ("aud", "tid", "oid", "scp", "appid")}
for name, value in fields.items():
    print(f"🎫 {name:<6} {value}")
assert fields["aud"] == "https://azure-api.net/aigateway" and fields["oid"] == my_object_id

### 4. Call the model without a key

`get_bearer_token_provider` returns a cached token and refreshes it before it expires. The OpenAI SDK calls it before
every request and sends `Authorization: Bearer <token>`. This is what
[infra/call_with_entra.py](infra/call_with_entra.py) does, with `DefaultAzureCredential` instead of
`AzureCliCredential`.

A new role assignment takes a few minutes to apply, so `utils.retry` retries `403` for up to 10 minutes.

In [ ]:
from azure.identity import get_bearer_token_provider
from openai import OpenAI

client = OpenAI(base_url=f"{lab.gateway_url}/default/models/openai/v1",
                api_key=get_bearer_token_provider(lab.credential, utils.AIGW_SCOPE))

completion = utils.retry(lambda: client.chat.completions.create(model=alias, messages=[
    {"role": "user", "content": "In one sentence, why use Microsoft Entra ID instead of API keys?"}]), statuses=(403,))

print(f"📨 {completion.choices[0].message.content}")
print(f"🔢 Tokens used: {completion.usage.total_tokens}")
print("✅ Keyless call succeeded")

### 5. Send a token for the wrong audience

A token for Azure Resource Manager is valid, but not for AI Gateway. The gateway answers `401`: authentication
failed, so the role was never checked.

In [ ]:
import requests

chat_url = f"{lab.gateway_url}/default/models/openai/v1/chat/completions"
payload  = {"model": alias, "max_tokens": 5, "messages": [{"role": "user", "content": "ping"}]}

def call_with(token):
    return requests.post(chat_url, json=payload, headers={"Authorization": f"Bearer {token}"}, timeout=60).status_code

arm_status = call_with(lab.token())   # a token for Azure Resource Manager
print(f"📨 ARM token:        HTTP {arm_status}")
assert arm_status == 401
print(f"📨 AI Gateway token: HTTP {call_with(lab.token(utils.AIGW_SCOPE))}")

### 6. Remove the role and try again

Remove your **AI Gateway Models User** assignments on the gateway, then call again. A deployment adds role
assignments but doesn't remove them, so the cell deletes them through Azure Resource Manager, like
`az role assignment delete`. Once the change applies, the gateway answers `403`: you're authenticated, but no role
allows you to invoke the model. In the playground, the same error offers **Review access** and
**Copy access request**.

The cell polls for up to 10 minutes. If you also hold a role at a broader scope, such as the resource group, the call
keeps working and the cell prints a warning instead.

In [ ]:
to_remove = {i.lower() for i in models_user_on_gateway()}
if assignment:
    to_remove.add(assignment["roleAssignmentId"].lower())
for assignment_id in to_remove:
    lab.delete(assignment_id, api_version=utils.ROLES_API)

def check():
    return call_with(lab.token(utils.AIGW_SCOPE))

status = utils.wait_for_status(check, 403)
print(f"📨 Without the role: HTTP {status}")
if status == 403:
    print("✅ The gateway denies callers without a role")
else:
    print("⚠️ Still allowed. Check for an inherited role in the list from step 2.")

### 7. Restore access

Deploy the template again and wait until the call returns `200`.

In [ ]:
assignment = grant()
status = utils.wait_for_status(check, 200)
print(f"📨 With the role: HTTP {status}")
assert status == 200

### 8. Clean up resources

Role assignments don't cost anything. The cell removes the assignment that this notebook created, and keeps it if you
had the role before you started. Set `keep_role = True` to keep it for other Entra-based labs.

In [ ]:
keep_role = False   # change to True to keep your role assignment

if keep_role or had_role_before:
    print("ℹ️ Kept your AI Gateway Models User assignment on the gateway")
else:
    lab.delete(assignment["roleAssignmentId"], api_version=utils.ROLES_API)

## Troubleshooting

- **`401` on every Entra call**: the token is expired, or it's for another audience, such as Azure Resource Manager.
  Request a token for `https://azure-api.net/aigateway/.default` and check `aud` in step 3.
- **`403` right after you assign the role**: role assignments take several minutes to apply. Wait and retry.
- **`400`**: the request sent both an `Authorization` header and an `api-key` header. Send one or the other.
- **`AuthorizationFailed` when you deploy the role assignment**: you need Owner, User Access Administrator, or Role
  Based Access Control Administrator on the gateway.
- **`RoleAssignmentExists`**: you already hold the role on the gateway under another assignment name, for example
  one you created in the portal. Rerun step 2: it skips the deployment when it finds the role.
- **`200` after you removed the role**: you hold a role at a broader scope, such as the resource group or
  subscription. List them with `az role assignment list --assignee <object-id> --all -o table`.
- **`CredentialUnavailableError`**: run `az login`, then restart the kernel.

## Next steps

- Secure the outbound boundary in [Call backends with managed identities](../12-backend-identity/README.md).
- Budget each Entra identity in [Enforce cost budgets with per-caller overrides](../07-cost-budgets/README.md).
- Docs: [Security and identity](https://ai.gateway.azure.com/docs/security-identity).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/11-entra-rbac/](https://azure.github.io/ai-gateway/labs/11-entra-rbac/).